# Stream a Response via our OpenAI Platform

This example shows how to stream a response from the OpenAI platform using our platform client.

## Setup


In [1]:
import os

from utils import setup_notebook

if not setup_notebook(required_keys=[
    "OPENAI_API_KEY",
]):
    raise ValueError("Failed to setup notebook, please check your .env file")

masked_key_id = os.getenv("OPENAI_API_KEY")[:5] + "*" * (
    len(os.getenv("OPENAI_API_KEY")) - 5
)
print(f"Using OpenAI API Key: {masked_key_id}")

Added paths to sys.path: ['/Users/dhiganthrao/Documents/Repositories/agent-platform/core/src', '/Users/dhiganthrao/Documents/Repositories/agent-platform/architectures/default/src', '/Users/dhiganthrao/Documents/Repositories/agent-platform/server/src']
Using OpenAI API Key: sk-sv********************************************************************************************************************************************************


In [4]:
from agent_platform.core.platforms.openai import OpenAIPlatformParameters
from agent_platform.core.platforms.openai.configs import OpenAIModelMap

# Configurations that will be used
default_model_map = OpenAIModelMap.default()
OpenAIModelMap.set_instance(default_model_map)

# Platform Parameters
parameters = OpenAIPlatformParameters(
    openai_api_key=os.getenv("OPENAI_API_KEY"),
)


## Create a Platform Client

In [5]:
from agent_platform.core.platforms.openai.client import OpenAIClient

openai_client = OpenAIClient(
    parameters=parameters,
)

## Create a Tool

In [6]:
from typing import Annotated

from agent_platform.core.tools import ToolDefinition


async def joke_judge(joke: Annotated[str, "A joke to judge"]) -> bool:
    """Judge if a joke is funny.

    Arguments:
        joke: A joke to judge.

    Returns:
        True if the joke is funny, False otherwise.
    """
    return False

joke_judge_tool = ToolDefinition.from_callable(joke_judge)

print(joke_judge_tool)



ToolDefinition(name='joke_judge', description='Judge if a joke is funny.\n\n    Arguments:\n        joke: A joke to judge.\n\n    Returns:\n        True if the joke is funny, False otherwise.', input_schema={'type': 'object', 'properties': {'joke': {'type': 'string', 'description': 'A joke to judge'}}, 'required': ['joke'], 'strict': True, 'additionalProperties': False}, function=<function joke_judge at 0x125330ea0>)


## Create a Prompt

In [7]:
from agent_platform.core.prompts import PromptTextContent, PromptUserMessage
from agent_platform.core.prompts.prompt import Prompt

user_prompt = PromptUserMessage(
    content=[PromptTextContent(
        text="Please use the joke_judge tool to "
        "judge if the following joke is funny: "
        "\"Why did the chicken cross the road?\"\n"
        "\"To get to the other side!\"",
        )],
)
general_prompt = Prompt(
    system_instruction="You are a helpful assistant.",
    messages=[user_prompt],
    tools=[joke_judge_tool],
)

await general_prompt.finalize_messages()
openai_prompt = await openai_client.converters.convert_prompt(general_prompt)

print(openai_prompt)



OpenAIPrompt(messages=[{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'Please use the joke_judge tool to judge if the following joke is funny: "Why did the chicken cross the road?"\n"To get to the other side!"'}], tools=[{'type': 'function', 'function': {'name': 'joke_judge', 'description': 'Judge if a joke is funny.\n\n    Arguments:\n        joke: A joke to judge.\n\n    Returns:\n        True if the joke is funny, False otherwise.', 'parameters': {'type': 'object', 'properties': {'joke': {'type': 'string', 'description': 'A joke to judge'}}, 'required': ['joke'], 'strict': True, 'additionalProperties': False}}}], max_tokens=4096, temperature=0.0, top_p=1.0)


## Request and Stream a Response

In [8]:
deltas = []
i = 0
async for delta in openai_client.generate_stream_response(
    openai_prompt, "gpt-4o-mini",
):
    print(f"CHUNK {i}: {delta!r}")
    deltas.append(delta)
    i += 1


{'model': 'gpt-4o-mini-2024-07-18', 'messages': [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'Please use the joke_judge tool to judge if the following joke is funny: "Why did the chicken cross the road?"\n"To get to the other side!"'}], 'tools': [{'type': 'function', 'function': {'name': 'joke_judge', 'description': 'Judge if a joke is funny.\n\n    Arguments:\n        joke: A joke to judge.\n\n    Returns:\n        True if the joke is funny, False otherwise.', 'parameters': {'type': 'object', 'properties': {'joke': {'type': 'string', 'description': 'A joke to judge'}}, 'required': ['joke'], 'strict': True, 'additionalProperties': False}}}], 'stream': True}
CHUNK 0: GenericDelta(op='add', path='/role', value='agent', from_=None)
CHUNK 1: GenericDelta(op='add', path='/content', value=[{'kind': 'tool_use', 'tool_call_id': 'call_3Wz71c0n1hQtLRXw0LJrT1jw', 'tool_name': 'joke_judge', 'tool_input_raw': ''}], from_=None)
CHUNK 2: GenericDelta(op=

## Reassemble

We now try to reassemble to full message to ensure it properly assembles into a `ResponseMessage`.

An implementation of this will need to be either in the AA or in a helper method attached to the kernel's `PlatformInterface`.

### First as a Dictionary

In [9]:
from pprint import pprint

from agent_platform.core.delta.combine_delta import combine_generic_deltas

assembled_dict = combine_generic_deltas(deltas)
pprint(assembled_dict)

{'additional_response_fields': {'id': 'chatcmpl-BIKbg1vvp72T6VmTuCxqtGOSwEzEb',
                                'model': 'gpt-4o-mini-2024-07-18'},
 'content': [{'kind': 'tool_use',
              'tool_call_id': 'call_3Wz71c0n1hQtLRXw0LJrT1jw',
              'tool_input_raw': '{"joke":"Why did the chicken cross the road? '
                                'To get to the other side!"}',
              'tool_name': 'joke_judge'}],
 'metadata': {'sema4ai_metadata': {'platform_name': 'openai'}},
 'raw_response': {'ResponseMetadata': {'HTTPStatusCode': 200,
                                       'RequestId': 'chatcmpl-BIKbg1vvp72T6VmTuCxqtGOSwEzEb',
                                       'RetryAttempts': 0},
                  'stream': None},
 'role': 'agent',
 'stop_reason': 'tool_calls'}


### Now as a ResponseMessage

In [10]:
from agent_platform.core.responses import ResponseMessage

response_message = ResponseMessage.model_validate(assembled_dict)
pprint(response_message)




ResponseMessage(content=[ResponseToolUseContent(kind='tool_use',
                                                tool_call_id='call_3Wz71c0n1hQtLRXw0LJrT1jw',
                                                tool_name='joke_judge',
                                                tool_input_raw='{"joke":"Why '
                                                               'did the '
                                                               'chicken cross '
                                                               'the road? To '
                                                               'get to the '
                                                               'other side!"}',
                                                _tool_input={'joke': 'Why did '
                                                                     'the '
                                                                     'chicken '
                                                                

# Continue the Conversation

## Execute the Tool

In [13]:
from agent_platform.core.responses import ResponseToolUseContent

tool_use_content_block = response_message.content[0]
assert isinstance(tool_use_content_block, ResponseToolUseContent)

is_funny = await joke_judge(tool_use_content_block.tool_input["joke"])

print(f"Is the joke funny? {'Yes' if is_funny else 'No'}")

Is the joke funny? No


## Create a ToolResponse Content Block

> **Note:** This skips writing to a thread, but we convert the tool content block of the `ResponseMessage` to a `ThreadToolUsageContent` so we can convert to `PromptToolResultContent`.

In [14]:
from agent_platform.core.prompts import PromptTextContent, PromptToolResultContent

assert isinstance(tool_use_content_block, ResponseToolUseContent)
tool_result_content = PromptToolResultContent(
    tool_name=tool_use_content_block.tool_name,
    tool_call_id=tool_use_content_block.tool_call_id,
    content=[PromptTextContent(text=str(is_funny))],
)

print(tool_result_content)


PromptToolResultContent(kind='tool_result', tool_name='joke_judge', tool_call_id='call_3Wz71c0n1hQtLRXw0LJrT1jw', content=[PromptTextContent(kind='text', text='False')], is_error=False)


## Extract the AI's Tool Usage Content Block

We must send back the AI's original tool use in the new request

In [16]:
from agent_platform.core.prompts import PromptToolUseContent
from agent_platform.core.thread import ThreadToolUsageContent

original_tool_use = response_message.content[0]
assert isinstance(original_tool_use, ResponseToolUseContent)

# Convert to ThreadToolUsageContent and then to PromptToolUseContent
ai_tool_use = ThreadToolUsageContent.from_response_tool_use(original_tool_use)
prompt_tool_use = PromptToolUseContent.from_thread_tool_usage(ai_tool_use)

pprint(prompt_tool_use)


PromptToolUseContent(kind='tool_use',
                     tool_call_id='call_3Wz71c0n1hQtLRXw0LJrT1jw',
                     tool_name='joke_judge',
                     tool_input_raw='{"joke":"Why did the chicken cross the '
                                    'road? To get to the other side!"}',
                     _tool_input={'joke': 'Why did the chicken cross the road? '
                                          'To get to the other side!'})


## Create new Prompt

In [17]:
from agent_platform.core.prompts import PromptAgentMessage

return_user_prompt = PromptUserMessage(
    content=[tool_result_content],
)
return_tool_use = PromptAgentMessage(
    content=[prompt_tool_use],
)
return_gen_prompt = Prompt(
    system_instruction="You are a helpful assistant.",
    messages=[user_prompt, return_tool_use, return_user_prompt],
    tools=[joke_judge_tool],
)
await return_gen_prompt.finalize_messages()
return_openai_prompt = await openai_client.converters.convert_prompt(
    return_gen_prompt,
)

pprint(return_openai_prompt)

OpenAIPrompt(messages=[{'content': 'You are a helpful assistant.',
                        'role': 'system'},
                       {'content': 'Please use the joke_judge tool to judge if '
                                   'the following joke is funny: "Why did the '
                                   'chicken cross the road?"\n'
                                   '"To get to the other side!"',
                        'role': 'user'},
                       {'content': '',
                        'role': 'assistant',
                        'tool_calls': [{'function': {'arguments': '{"joke": '
                                                                  '"Why did '
                                                                  'the chicken '
                                                                  'cross the '
                                                                  'road? To '
                                                                  'get to the '


## Generate a new Response (Non-Streaming)

In [18]:
response = await openai_client.generate_response(
    return_openai_prompt,
    "gpt-4o-mini",
)

pprint(response)

{'model': 'gpt-4o-mini-2024-07-18', 'messages': [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'Please use the joke_judge tool to judge if the following joke is funny: "Why did the chicken cross the road?"\n"To get to the other side!"'}, {'role': 'assistant', 'content': '', 'tool_calls': [{'id': 'call_3Wz71c0n1hQtLRXw0LJrT1jw', 'type': 'function', 'function': {'name': 'joke_judge', 'arguments': '{"joke": "Why did the chicken cross the road? To get to the other side!"}'}}]}, {'role': 'tool', 'tool_call_id': 'call_3Wz71c0n1hQtLRXw0LJrT1jw', 'content': 'False'}], 'tools': [{'type': 'function', 'function': {'name': 'joke_judge', 'description': 'Judge if a joke is funny.\n\n    Arguments:\n        joke: A joke to judge.\n\n    Returns:\n        True if the joke is funny, False otherwise.', 'parameters': {'type': 'object', 'properties': {'joke': {'type': 'string', 'description': 'A joke to judge'}}, 'required': ['joke'], 'strict': True, 'additiona